In [ ]:
import numpy as np
import xarray as xr
import tensorflow as tf
import joblib
import sys
import os
sys.path.append(os.path.abspath(".."))
from src.config import load_paths
from model.model import read_scale_data
from evaluation.flux_energy import parsavel_avarge_cofficents
from evaluation.flux_energy import zonally_avarge
import matplotlib.pyplot as plt

In [ ]:
Y_scaled = np.zeros((20480, 144))
Y_predicted = np.zeros((20480, 144))
omega_hat = np.zeros((20480, 1, 6, 6, 4))
cgz = np.zeros((20480, 1, 6, 6, 4))
ntri_dict = {}
for tag in  ["land", "ocean"]:
				paths = load_paths()
				ml_model_dir = paths["ml_model_dir"]
				fname_model = f"cnn_coarsed_{tag}_2.keras"
				model = tf.keras.models.load_model(
												ml_model_dir + fname_model,
												custom_objects=False, compile=False)
				season_tag = "Nov"
				data_base_dir = paths[f"data_base_dir_{season_tag}"]
				ds_name = f"corsed_action_{tag}_2.pkl"
				scaler = joblib.load(paths["save_dir"] + "/scaler/" + f"scaler_X_{tag}.pkl")
				ds = joblib.load(data_base_dir + ds_name)
				X_scaler = scaler.transform(ds["X"][:, 0, :])
				Y_s = np.log(ds["Y"][:, 0, :] + 1).reshape(-1, 144)
				cgz[ds["ntri"], ...] = ds["cgz"]
				omega_hat[ds["ntri"], ...] = ds["omega_hat"]
				ntri_dict[f"ntri_{tag}"] = ds["ntri"]
				Y_scaled[ds["ntri"], ...] = np.nan_to_num(Y_s, 0.0)
				Y_predicted[ds["ntri"]] = model.predict(X_scaler)

In [ ]:
for tri_tag in ["land", "ocean"]:
    MSE = np.mean((Y_scaled[ntri_dict[f"ntri_{tri_tag}"]]
                    - Y_predicted[ntri_dict[f"ntri_{tri_tag}"]])**2, axis=0)
    var = np.var(Y_scaled[ntri_dict[f"ntri_{tri_tag}"]], axis=0)
    R2 = 1 - MSE/var
    print(f"MSE {tri_tag} : {MSE.mean()}")
    print(f"R2 {tri_tag} : {R2.mean()}")

In [ ]:
N_test = np.expm1(Y_scaled)
N_pred = np.expm1(Y_predicted)
for tag_land_ocean in ["land", "ocean"]:
				MAE    = np.mean(np.abs(N_test[ntri_dict[f"ntri_{tag_land_ocean}"]]
							 - N_pred[ntri_dict[f"ntri_{tag_land_ocean}"]]), axis=0)

				save_dir = paths["save_dir"]

				(_, _, y_train, _, _, _, _, _, _, _, _, 
									_, _, _, _, _, _) = read_scale_data(save_dir, land_ocean_tag=tag_land_ocean, coarse_action_dir_jan=paths["coarse_action_dir_jan"],
                                                                        physics=False)
				N_train = np.expm1(y_train)
				std    = np.std(N_train, axis=0)
				NMAE   = MAE/std
				del y_train
				N_base_line = np.mean(N_train, axis=0)
				mae_base_line = np.mean(np.abs(N_test[ntri_dict[f"ntri_{tag_land_ocean}"]]
								    - N_base_line), axis=0)
				score = 1 - MAE/mae_base_line
				del N_train
				print(f"MAE {tag_land_ocean} : {MAE.mean()}")
				print(f"NMAE {tag_land_ocean} : {NMAE.mean()}")
				print(f"score {tag_land_ocean} : {score.mean()}")

In [ ]:
tag_season = "Nov"
paths = load_paths()
coarse_action_dir      = paths[f"coarse_action_dir_{tag_season}"]
with xr.open_dataset(coarse_action_dir + "coarse_gw_spectrum_group_1_fileslide_0.nc") as ds1:
        k_lunch = ds1.k.data
        l_lunch = ds1.l.data
        m_lunch = ds1.m.data
print(f"k is :{k_lunch}")
print(f"l is :{l_lunch}")
print(f"m is :{m_lunch}")

In [ ]:
omega_hat = omega_hat.reshape(-1, 144)
N_test = np.expm1(Y_scaled)
N_pred = np.expm1(Y_predicted)
E_pred = np.sum(N_pred*omega_hat, axis=1)
E_test = np.sum(N_test*omega_hat, axis=1)
bias      = (np.mean(E_pred) - np.mean(E_test))/np.mean(E_test)
r         = np.corrcoef(E_test, E_pred)[0, 1]
print(f"bias of energy : {bias}")
print(f"corrolation {r}")

In [ ]:
import os
import numpy as np
import xarray as xr
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from scipy.stats import spearmanr


def plot_global_map_pair(
    E_true,
    E_pred,
    save_dir,
    filename="E_test_vs_model_nov",
    month="November",
    cmap="magma_r",
    pct=(1, 99),          # robust colour limits from true and predicted combined
    fontsize=10,
):
    gridfil = xr.open_dataset("/work/bb1097/b381585/data/icon_grid_0012_R02B04_G.nc")
    lon = np.rad2deg(gridfil.clon_vertices.values)
    lat = np.rad2deg(gridfil.clat_vertices.values)
    gridfil.close()
    lon = np.where(lon < -180, lon + 360, lon)
    lon = np.where(lon > 180, lon - 360, lon)
    tris = np.stack([lon, lat], axis=-1)

    lt, lp = np.log10(E_true), np.log10(E_pred)
    vmin, vmax = np.nanpercentile(np.concatenate([lt, lp]), pct)
    norm = plt.Normalize(vmin, vmax)
    cmo = plt.get_cmap(cmap)
    print(f"{month}: colour range log10E = [{vmin:.2f}, {vmax:.2f}]")

    fig, axes = plt.subplots(
        2, 1, figsize=(5.0, 5.4),
        subplot_kw=dict(projection=ccrs.PlateCarree()),
        constrained_layout=True,
    )

    for row, (var, label) in enumerate([(lt, "True"), (lp, "Predicted")]):
        ax = axes[row]
        ax.set_global()
        ax.add_collection(PolyCollection(
            tris, fc=cmo(norm(var)), edgecolors="none",
            transform=ccrs.Geodetic(), zorder=0))
        ax.coastlines(linewidth=0.5, zorder=2)

        gl = ax.gridlines(draw_labels=True, linewidth=0.3,
                          color="dimgray", alpha=0.45, zorder=2)
        gl.top_labels = False
        gl.right_labels = False
        gl.bottom_labels = (row == 1)
        gl.xlabel_style = gl.ylabel_style = {"size": fontsize - 3}
        ax.set_title(f"{label} ({month})", fontsize=fontsize, pad=3)

    sm = plt.cm.ScalarMappable(cmap=cmo, norm=norm)
    sm.set_array([])
    cb = fig.colorbar(sm, ax=axes, orientation="horizontal",
                      fraction=0.05, pad=0.04, shrink=0.9, extend="both")
    cb.set_label(r"$\log_{10}E$ ($E$ in kg m$^{-1}$ s$^{-2}$)", fontsize=fontsize)
    cb.ax.tick_params(labelsize=fontsize - 1)

    os.makedirs(save_dir, exist_ok=True)
    for ext in ("png", "pdf"):
        plt.savefig(os.path.join(save_dir, f"{filename}.{ext}"),
                    dpi=400, bbox_inches="tight")
    plt.show()


# E_test, E_pred: time-averaged energy per triangle (positive values)
plot_global_map_pair(
    E_test, E_pred,
    save_dir="/work/bb1097/b383521/plots/paper_plots/",
    filename="E_test_vs_model_nov",
    month="November",
)

# statistics for the text (same definitions as for January and August)
print("r (linear)      :", np.corrcoef(E_test, E_pred)[0, 1])
print("r (log10)       :", np.corrcoef(np.log10(E_test), np.log10(E_pred))[0, 1])
print("Spearman        :", spearmanr(E_test, E_pred)[0])
print("rel. bias (mean):", np.mean(E_pred) / np.mean(E_test) - 1)
print("mean log10 bias :", np.mean(np.log10(E_pred / E_test)))

In [ ]:
ntri = ds["ntri"]
k = k_lunch[None, None, :, None, None]
l = l_lunch[None, None, None, :, None]
uw_test = np.sum(-N_test.reshape(20480, 1, 6, 6, 4)*cgz*k, axis=(2, 3, 4))
uw_pred = np.sum(-N_pred.reshape(20480, 1, 6, 6, 4)*cgz*k, axis=(2, 3, 4))
vw_test =  np.sum(-N_test.reshape(20480, 1, 6, 6, 4)*cgz*l, axis=(2, 3, 4))
vw_pred =  np.sum(-N_pred.reshape(20480, 1, 6, 6, 4)*cgz*l, axis=(2, 3, 4))

In [ ]:
pars_coff = parsavel_avarge_cofficents(decomposed_dir=paths["decomposed_dir"],
                                        tri_list_dir=paths["tri_list_dir"])
ntri = ds["ntri"]
k = k_lunch[None, None, :, None, None]
l = l_lunch[None, None, None, :, None]
uw_test = np.sum(N_test.reshape(20480, 1, 6, 6, 4)*cgz*k, axis=(2, 3, 4))*pars_coff
uw_pred = np.sum(N_pred.reshape(20480, 1, 6, 6, 4)*cgz*k, axis=(2, 3, 4))*pars_coff
vw_test =  np.sum(N_test.reshape(20480, 1, 6, 6, 4)*cgz*l, axis=(2, 3, 4))*pars_coff
vw_pred =  np.sum(N_pred.reshape(20480, 1, 6, 6, 4)*cgz*l, axis=(2, 3, 4))*pars_coff

In [ ]:
uw_test_avrage, lat_bins = zonally_avarge(var=uw_test*1000, 
                                grid_file=paths["grid_file_low"])
uw_pred_avrage, lat_bins = zonally_avarge(var=uw_pred*1000, 
                                grid_file=paths["grid_file_low"])
vw_pred_avrage, lat_bins = zonally_avarge(var=vw_pred*1000, 
                                grid_file=paths["grid_file_low"])
vw_test_avrage, lat_bins = zonally_avarge(var=vw_test*1000, 
                                grid_file=paths["grid_file_low"])


lat = 0.5 * (lat_bins[:-1] + lat_bins[1:])
def plot_zonal_mean_pair(
    lat,
    uw_test,
    uw_pred,
    vw_test,
    vw_pred,
    savename="zonal_mean_uw_vw",
    fontsize=11,  # Standard paper font size (9–11 pt)
):
    # Standard 2-column paper figure dimensions
    fig, axes = plt.subplots(
        1, 2, figsize=(7.0, 3.2), sharey=False, constrained_layout=True
    )

    datasets = [
        (r"$\rho\overline{u'w'}$ [mPa]", uw_test, uw_pred, axes[0]),
        (r"$\rho\overline{v'w'}$ [mPa]", vw_test, vw_pred, axes[1]),
    ]

    handles = []

    for ylabel, true_vals, pred_vals, ax in datasets:
        ax.axhline(0, color="gray", lw=0.8, ls=":", alpha=0.7)

        (l1,) = ax.plot(lat, true_vals, "-", color="black", lw=1.8, label="True")
        (l2,) = ax.plot(
            lat, pred_vals, "--", color="firebrick", lw=1.8, label="Predicted"
        )

        ax.set_xlim(lat.min(), lat.max())

        # Clean degree-formatted latitude ticks
        xticks = np.arange(-90, 91, 30)
        xticklabels = [
            f"{abs(x)}°S" if x < 0 else (f"{x}°N" if x > 0 else "0°")
            for x in xticks
        ]
        ax.set_xticks(xticks)
        ax.set_xticklabels(xticklabels, fontsize=fontsize - 1)

        ax.tick_params(axis="both", labelsize=fontsize - 1)
        ax.grid(alpha=0.25, lw=0.5)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.set_ylabel(ylabel, fontsize=fontsize)

        if ax == axes[0]:
            handles = [l1, l2]

    # Centered figure-level x-axis label
    fig.supxlabel("Latitude", fontsize=fontsize)

    # Clean upper legend
    fig.legend(
        handles=handles,
        labels=["True", "Predicted"],
        loc="lower center",
        ncol=2,
        bbox_to_anchor=(0.5, 1.01),
        fontsize=fontsize - 1,
        frameon=False,
    )

    plt.savefig(
        f"/work/bb1097/b383521/plots/paper_plots/{savename}.png",
        dpi=400,
        bbox_inches="tight",
    )
    plt.savefig(
        f"/work/bb1097/b383521/plots/paper_plots/{savename}.pdf",
        dpi=400,
        bbox_inches="tight",
    )
    plt.show()
plot_zonal_mean_pair(
    lat,
    uw_test_avrage,
    uw_pred_avrage,
    vw_test_avrage,
    vw_pred_avrage,
    savename="zonal_mean_uw_vw",
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# zonal means (Pa -> mPa), as before
uw_test_avrage, lat_bins = zonally_avarge(var=uw_test * 1000,
                                          grid_file=paths["grid_file_low"])
uw_pred_avrage, lat_bins = zonally_avarge(var=uw_pred * 1000,
                                          grid_file=paths["grid_file_low"])
vw_test_avrage, lat_bins = zonally_avarge(var=vw_test * 1000,
                                          grid_file=paths["grid_file_low"])
vw_pred_avrage, lat_bins = zonally_avarge(var=vw_pred * 1000,
                                          grid_file=paths["grid_file_low"])

lat = 0.5 * (lat_bins[:-1] + lat_bins[1:])


def plot_zonal_mean_pair(
    lat,
    uw_test, uw_pred,
    vw_test, vw_pred,
    savename="zonal_mean_uw_vw_nov",
    show_r=True,
    fontsize=10,
):
    panels = [
        (r"$\rho\overline{u'w'}$ (mPa)", uw_test, uw_pred),
        (r"$\rho\overline{v'w'}$ (mPa)", vw_test, vw_pred),
    ]

    fig, axes = plt.subplots(1, 2, figsize=(5.2, 3.4), sharey=True,
                             constrained_layout=True)

    yticks = np.arange(-90, 91, 30)
    yticklabels = [f"{abs(y)}°S" if y < 0 else (f"{y}°N" if y > 0 else "0°")
                   for y in yticks]

    handles = None
    for i, (ax, (label, tr, pr)) in enumerate(zip(axes, panels)):
        m = 1.08 * np.nanmax(np.abs(np.concatenate([tr, pr])))

        ax.axvline(0, color="gray", lw=0.8, ls=":", alpha=0.8)
        ax.fill_betweenx(lat, tr, pr, color="firebrick", alpha=0.12, lw=0)
        (l1,) = ax.plot(tr, lat, "-", color="black", lw=1.6, label="True")
        (l2,) = ax.plot(pr, lat, "--", color="firebrick", lw=1.6, label="Predicted")
        handles = [l1, l2]

        ax.set_xlim(-m, m)
        ax.set_ylim(lat.min(), lat.max())
        ax.set_yticks(yticks)
        ax.set_yticklabels(yticklabels)
        ax.tick_params(labelsize=fontsize - 1)
        ax.grid(alpha=0.25, lw=0.5)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.set_xlabel(label, fontsize=fontsize)
        if i == 0:
            ax.set_ylabel("Latitude", fontsize=fontsize)

        if show_r:
            r = np.corrcoef(tr, pr)[0, 1]
            ax.text(0.04, 0.03, f"$r={r:.2f}$", transform=ax.transAxes,
                    fontsize=fontsize - 2, ha="left", va="bottom",
                    bbox=dict(fc="w", ec="none", alpha=0.8, pad=1.5))

    fig.legend(handles=handles, labels=["True", "Predicted"], loc="lower center",
               ncol=2, bbox_to_anchor=(0.5, 1.0), fontsize=fontsize - 1,
               frameon=False)

    for ext in ("png", "pdf"):
        plt.savefig(f"/work/bb1097/b383521/plots/paper_plots/{savename}.{ext}",
                    dpi=400, bbox_inches="tight")
    plt.show()


def to_1d(a, n_lat):
    a = np.asarray(a)
    if a.ndim == 1:
        return a
    # average over the time axis, whichever axis is not the latitude axis
    lat_axis = a.shape.index(n_lat)
    return np.nanmean(a, axis=1 - lat_axis if a.ndim == 2 else None)

n_lat = len(lat)
uw_test_1d = to_1d(uw_test_avrage, n_lat)
uw_pred_1d = to_1d(uw_pred_avrage, n_lat)
vw_test_1d = to_1d(vw_test_avrage, n_lat)
vw_pred_1d = to_1d(vw_pred_avrage, n_lat)

print(uw_test_1d.shape, lat.shape)   # both should be (n_lat,)

plot_zonal_mean_pair(
    lat,
    uw_test_1d, uw_pred_1d,
    vw_test_1d, vw_pred_1d,
    savename="zonal_mean_uw_vw_nov",
)

for name, t, p in [("uw Nov", uw_test_1d, uw_pred_1d),
                   ("vw Nov", vw_test_1d, vw_pred_1d)]:
    for k, label in [(np.argmax(t), "max"), (np.argmin(t), "min")]:
        print(f"{name} {label}: lat={lat[k]:.1f}  true={t[k]:.3f}  pred={p[k]:.3f}  "
              f"underestimate={100 * (1 - p[k] / t[k]):.0f}%")


In [ ]:
import os
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from matplotlib.collections import PolyCollection
from matplotlib.colors import TwoSlopeNorm


def plot_global_map_flux_pair(
    var_top,
    var_bottom,
    save_dir,
    filename="global_momentum_flux_map_pair",
    title_top="True",
    title_bottom="Predicted",
    cbar_label=r"$\rho\overline{u'w'}$ [mPa]",
    cmap="RdBu_r",
    vmin=None,
    vmax=None,
    percentile_clip=True,
    p_low=2,
    p_high=98,
    symmetric=True,
    fontsize=11,
):
    var_top = np.asarray(var_top, dtype=float)
    var_bottom = np.asarray(var_bottom, dtype=float)

    gridfil = xr.open_dataset(
        "/work/bb1097/b381585/data/icon_grid_0012_R02B04_G.nc"
    )
    clon_vertices = np.rad2deg(gridfil.clon_vertices.values)
    clat_vertices = np.rad2deg(gridfil.clat_vertices.values)
    gridfil.close()

    clon_vertices = np.where(
        clon_vertices < -180.0, clon_vertices + 360.0, clon_vertices
    )
    clon_vertices = np.where(
        clon_vertices > 180.0, clon_vertices - 360.0, clon_vertices
    )
    triangles = np.stack([clon_vertices, clat_vertices], axis=-1)

    # ---------------------------
    # Shared color range across BOTH panels
    # ---------------------------
    if vmin is None or vmax is None:
        combined = np.concatenate(
            [var_top[np.isfinite(var_top)], var_bottom[np.isfinite(var_bottom)]]
        )
        if percentile_clip:
            p_lo_val, p_hi_val = np.nanpercentile(combined, [p_low, p_high])
            if symmetric:
                vmax_abs = max(abs(p_lo_val), abs(p_hi_val))
                vmin, vmax = -vmax_abs, vmax_abs
            else:
                vmin, vmax = p_lo_val, p_hi_val
        else:
            vmin, vmax = np.nanmin(combined), np.nanmax(combined)

    print(f"Using shared color range: vmin={vmin:.4f}, vmax={vmax:.4f}")

    cmap_obj = plt.get_cmap(cmap)
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)

    # 2 rows, 1 column vertical stack layout (Standard paper width 7.0 in)
    fig, axes = plt.subplots(
        2,
        1,
        figsize=(7.0, 5.8),
        subplot_kw=dict(projection=ccrs.PlateCarree()),
        constrained_layout=True,
    )

    datasets = [
        (var_top, title_top, axes[0]),
        (var_bottom, title_bottom, axes[1]),
    ]

    for var, subtitle, ax in datasets:
        colors = cmap_obj(norm(var))
        colors[~np.isfinite(var)] = [1, 1, 1, 0]

        ax.set_global()
        ax.coastlines(linewidth=0.6, zorder=2)

        gl = ax.gridlines(
            draw_labels=True,
            linewidth=0.4,
            color="dimgray",
            alpha=0.45,
            zorder=2,
        )
        gl.top_labels = False
        gl.right_labels = False
        gl.xlabel_style = {"size": fontsize - 2}
        gl.ylabel_style = {"size": fontsize - 2}

        coll = PolyCollection(
            triangles,
            fc=colors,
            edgecolors="none",
            linewidth=0.05,
            transform=ccrs.Geodetic(),
            zorder=0,
        )
        ax.add_collection(coll)

        # Unbolded subpanel title
        ax.set_title(subtitle, fontsize=fontsize, pad=4)

    sm = plt.cm.ScalarMappable(cmap=cmap_obj, norm=norm)
    sm.set_array([])

    # Vertical colorbar on the right with compact width and padding
    cbar = fig.colorbar(
        sm,
        ax=axes,
        orientation="vertical",
        fraction=0.03,
        pad=0.03,
        shrink=0.85,
        extend="both",
    )
    cbar.set_label(cbar_label, fontsize=fontsize, fontweight="normal")
    cbar.ax.tick_params(labelsize=fontsize - 1)

    os.makedirs(save_dir, exist_ok=True)
    outpath_png = os.path.join(save_dir, f"{filename}.png")
    outpath_pdf = os.path.join(save_dir, f"{filename}.pdf")

    plt.savefig(outpath_png, dpi=400, bbox_inches="tight")
    plt.savefig(outpath_pdf, dpi=400, bbox_inches="tight")
    plt.show()
    print(f"--> saved to {outpath_png} and {outpath_pdf}")
# --- uw: January ---
plot_global_map_flux_pair(
    uw_test*1000,
    uw_pred*1000,
    save_dir="/work/bb1097/b383521/plots/paper_plots/",
    filename="uw_momentum_flux_mpa_nov",
    title_top="True",
    title_bottom="Predicted",
    cbar_label=r"$\rho\overline{u'w'}$ [mPa]",
    symmetric=True,
)

plot_global_map_flux_pair(
    vw_test*1000,
    vw_pred*1000,
    save_dir="/work/bb1097/b383521/plots/paper_plots/",
    filename="vw_momentum_flux_mpa_nov",
    title_top="True",
    title_bottom="Predicted",
    cbar_label=r"$\rho\overline{v'w'}$ [mPa]",
    symmetric=True,
)

## compute correlation for uw and vw in nov
r_uw_nov = np.corrcoef(uw_test[:, 0]*1000, uw_pred[:, 0]*1000)[0, 1]
r_vw_nov = np.corrcoef(vw_test[:, 0]*1000, vw_pred[:, 0]*1000)[0,	1]
print(f"uw: November:  corrolation = {r_uw_nov:.3f}")
print(f"vw: November: corrolation = {r_vw_nov:.3f}")

In [ ]:
import os
import numpy as np
import xarray as xr
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from matplotlib.colors import SymLogNorm, TwoSlopeNorm


def plot_flux_map_nov(
    uw_true, uw_pred, vw_true, vw_pred,
    save_dir,
    filename="momentum_flux_nov",
    month="November",
    cmap="RdBu_r",
    p_clip=99,              # symmetric colour limit from this percentile of |F|
    scale="symlog",         # "symlog" or "linear"
    linthresh_frac=0.08,    # linear region = +-linthresh_frac * vmax (symlog only)
    fontsize=10,
):
    gridfil = xr.open_dataset("/work/bb1097/b381585/data/icon_grid_0012_R02B04_G.nc")
    lon = np.rad2deg(gridfil.clon_vertices.values)
    lat = np.rad2deg(gridfil.clat_vertices.values)
    gridfil.close()
    lon = np.where(lon < -180, lon + 360, lon)
    lon = np.where(lon > 180, lon - 360, lon)
    tris = np.stack([lon, lat], axis=-1)

    # squeeze (n_tri, 1) -> (n_tri,)
    f = lambda a: np.asarray(a, float).reshape(-1)
    columns = {
        r"$\rho\overline{u'w'}$ (mPa)": (f(uw_true), f(uw_pred)),
        r"$\rho\overline{v'w'}$ (mPa)": (f(vw_true), f(vw_pred)),
    }

    fig, axes = plt.subplots(
        2, 2, figsize=(7.2, 4.8),
        subplot_kw=dict(projection=ccrs.PlateCarree()),
        constrained_layout=True,
    )
    cmo = plt.get_cmap(cmap)

    for col, (label, (ft, fp)) in enumerate(columns.items()):
        both = np.concatenate([ft[np.isfinite(ft)], fp[np.isfinite(fp)]])
        vmax = np.percentile(np.abs(both), p_clip)

        if scale == "symlog":
            linthresh = linthresh_frac * vmax
            norm = SymLogNorm(linthresh=linthresh, linscale=0.7,
                              vmin=-vmax, vmax=vmax, base=10)
            print(f"{label}: range +-{vmax:.3f} mPa, linthresh {linthresh:.3f} mPa")
        else:
            norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)
            print(f"{label}: range +-{vmax:.3f} mPa (linear)")

        for row, (var, name) in enumerate([(ft, "True"), (fp, "Predicted")]):
            ax = axes[row, col]
            colors = cmo(norm(var))
            colors[~np.isfinite(var)] = [1, 1, 1, 0]

            ax.set_global()
            ax.add_collection(PolyCollection(
                tris, fc=colors, edgecolors="none",
                transform=ccrs.Geodetic(), zorder=0))
            ax.coastlines(linewidth=0.5, zorder=2)

            gl = ax.gridlines(draw_labels=True, linewidth=0.3,
                              color="dimgray", alpha=0.45, zorder=2)
            gl.top_labels = False
            gl.right_labels = False
            gl.left_labels = (col == 0)
            gl.bottom_labels = (row == 1)
            gl.xlabel_style = gl.ylabel_style = {"size": fontsize - 3}
            ax.set_title(f"{name} ({month})", fontsize=fontsize, pad=3)

        sm = plt.cm.ScalarMappable(cmap=cmo, norm=norm)
        sm.set_array([])
        cb = fig.colorbar(sm, ax=axes[:, col], orientation="horizontal",
                          fraction=0.06, pad=0.04, shrink=0.9, extend="both")
        if scale == "symlog":
            t = np.array([0.1, 0.3, 1.0]) * vmax
            ticks = np.concatenate([-t[::-1], [0], t])
            cb.set_ticks(ticks)
            cb.set_ticklabels([f"{v:.2g}" for v in ticks])
        cb.set_label(label, fontsize=fontsize)
        cb.ax.tick_params(labelsize=fontsize - 1)

    os.makedirs(save_dir, exist_ok=True)
    for ext in ("png", "pdf"):
        plt.savefig(os.path.join(save_dir, f"{filename}.{ext}"),
                    dpi=400, bbox_inches="tight")
    plt.show()


plot_flux_map_nov(
    uw_test * 1000, uw_pred * 1000,
    vw_test * 1000, vw_pred * 1000,
    save_dir="/work/bb1097/b383521/plots/paper_plots/",
    filename="momentum_flux_nov",
)

# correlations (Pearson, same as in the January/August table)
r_uw = np.corrcoef(np.ravel(uw_test), np.ravel(uw_pred))[0, 1]
r_vw = np.corrcoef(np.ravel(vw_test), np.ravel(vw_pred))[0, 1]
print(f"November: uw r = {r_uw:.3f}, vw r = {r_vw:.3f}")

In [ ]:
print("uw_test shape:", uw_test.shape)
print("uw_pred shape:", uw_pred.shape)

print("vw_test shape:", vw_test.shape)
print("vw_pred shape:", vw_pred.shape)

print("uw number of elements:", np.size(uw_test))
print("vw number of elements:", np.size(vw_test))

print("UW NaNs:", np.isnan(uw_test).sum(), np.isnan(uw_pred).sum())
print("VW NaNs:", np.isnan(vw_test).sum(), np.isnan(vw_pred).sum())

In [ ]:
import os
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from matplotlib.collections import PolyCollection
from matplotlib.colors import TwoSlopeNorm


def plot_global_map_flux_pair(
    var_top,
    var_bottom,
    save_dir,
    filename="global_momentum_flux_map_pair",
    title_top="True",
    title_bottom="Predicted",
    cbar_label=r"$\rho\overline{u'w'}$ [mPa]",
    cmap="RdBu_r",
    vmin=None,
    vmax=None,
    percentile_clip=True,
    p_low=2,
    p_high=98,
    symmetric=True,
    fontsize=11,
):
    var_top = np.asarray(var_top, dtype=float)
    var_bottom = np.asarray(var_bottom, dtype=float)

    gridfil = xr.open_dataset(
        "/work/bb1097/b381585/data/icon_grid_0012_R02B04_G.nc"
    )
    clon_vertices = np.rad2deg(gridfil.clon_vertices.values)
    clat_vertices = np.rad2deg(gridfil.clat_vertices.values)
    gridfil.close()

    clon_vertices = np.where(
        clon_vertices < -180.0, clon_vertices + 360.0, clon_vertices
    )
    clon_vertices = np.where(
        clon_vertices > 180.0, clon_vertices - 360.0, clon_vertices
    )
    triangles = np.stack([clon_vertices, clat_vertices], axis=-1)

    # ---------------------------
    # Shared color range across BOTH panels
    # ---------------------------
    if vmin is None or vmax is None:
        combined = np.concatenate(
            [var_top[np.isfinite(var_top)], var_bottom[np.isfinite(var_bottom)]]
        )
        if percentile_clip:
            p_lo_val, p_hi_val = np.nanpercentile(combined, [p_low, p_high])
            if symmetric:
                vmax_abs = max(abs(p_lo_val), abs(p_hi_val))
                vmin, vmax = -vmax_abs, vmax_abs
            else:
                vmin, vmax = p_lo_val, p_hi_val
        else:
            vmin, vmax = np.nanmin(combined), np.nanmax(combined)

    print(f"Using shared color range: vmin={vmin:.4f}, vmax={vmax:.4f}")

    cmap_obj = plt.get_cmap(cmap)
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)

    # 1 row, 2 columns side-by-side layout (wide format for two panels in a row)
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(11.0, 3.4),
        subplot_kw=dict(projection=ccrs.PlateCarree()),
        constrained_layout=True,
    )

    datasets = [
        (var_top, title_top, axes[0]),
        (var_bottom, title_bottom, axes[1]),
    ]

    for var, subtitle, ax in datasets:
        colors = cmap_obj(norm(var))
        colors[~np.isfinite(var)] = [1, 1, 1, 0]

        ax.set_global()
        ax.coastlines(linewidth=0.6, zorder=2)

        gl = ax.gridlines(
            draw_labels=True,
            linewidth=0.4,
            color="dimgray",
            alpha=0.45,
            zorder=2,
        )
        gl.top_labels = False
        gl.right_labels = False
        gl.xlabel_style = {"size": fontsize - 2}
        gl.ylabel_style = {"size": fontsize - 2}

        coll = PolyCollection(
            triangles,
            fc=colors,
            edgecolors="none",
            linewidth=0.05,
            transform=ccrs.Geodetic(),
            zorder=0,
        )
        ax.add_collection(coll)

        # Unbolded subpanel title
        ax.set_title(subtitle, fontsize=fontsize, pad=4)

    sm = plt.cm.ScalarMappable(cmap=cmap_obj, norm=norm)
    sm.set_array([])

    # Single vertical colorbar to the right of both row panels
    cbar = fig.colorbar(
        sm,
        ax=axes,
        orientation="vertical",
        fraction=0.025,
        pad=0.02,
        shrink=0.85,
        extend="both",
    )
    cbar.set_label(cbar_label, fontsize=fontsize, fontweight="normal")
    cbar.ax.tick_params(labelsize=fontsize - 1)

    os.makedirs(save_dir, exist_ok=True)
    outpath_png = os.path.join(save_dir, f"{filename}.png")
    outpath_pdf = os.path.join(save_dir, f"{filename}.pdf")

    plt.savefig(outpath_png, dpi=400, bbox_inches="tight")
    plt.savefig(outpath_pdf, dpi=400, bbox_inches="tight")
    plt.show()
    print(f"--> saved to {outpath_png} and {outpath_pdf}")


# --- uw: January ---
plot_global_map_flux_pair(
    uw_test * 1000,
    uw_pred * 1000,
    save_dir="/work/bb1097/b383521/plots/paper_plots/",
    filename="uw_momentum_flux_mpa_nov",
    title_top="True",
    title_bottom="Predicted",
    cbar_label=r"$\rho\overline{u'w'}$ [mPa]",
    symmetric=True,
)

plot_global_map_flux_pair(
    vw_test * 1000,
    vw_pred * 1000,
    save_dir="/work/bb1097/b383521/plots/paper_plots/",
    filename="vw_momentum_flux_mpa_nov",
    title_top="True",
    title_bottom="Predicted",
    cbar_label=r"$\rho\overline{v'w'}$ [mPa]",
    symmetric=True,
)